In [ ]:
pip install transformers datasets evaluate accelerate

In [ ]:
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')

In [ ]:
print(dataset)

In [ ]:
print(dataset['train'][0])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_datasets)
print(tokenized_datasets['train'][0].keys())

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

In [ ]:
import numpy as np
import evaluate

# Subset chico para validar que todo funciona
small_train = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))
small_eval = tokenized_datasets['test'].shuffle(seed=42).select(range(500))

In [ ]:
accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
training_args_full = TrainingArguments(
    output_dir='./results-full',
    eval_strategy='epoch',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_steps=100,
)

trainer_full = Trainer(
    model=model,
    args=training_args_full,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

trainer_full.train()